# 01: Data Exploration — VSR Dataset Analysis

**Goal**: Understand the Visual Spatial Reasoning (VSR) dataset before training models.

Key tasks:
- Load VSR dataset from HuggingFace
- Analyze relation type distribution
- Check label balance (True/False per relation)
- Examine caption length and image availability
- Display sample images with captions

Reference: Liu et al. (2022) "Visual Spatial Reasoning" https://arxiv.org/abs/2205.00363

## 1. Setup and Environment Configuration

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from dotenv import load_dotenv
import os

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent))

# Load environment config
load_dotenv()
CACHE_DIR = Path(os.getenv("CACHE_DIR", "../results/embeddings"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Set seeds for reproducibility
np.random.seed(42)

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print(f"Cache directory: {CACHE_DIR}")

## 2. Load VSR Dataset

In [ ]:
from datasets import load_dataset

# Load VSR from HuggingFace
print("Loading VSR dataset (random split)...")
vsr = load_dataset("cambridgeltl/vsr_random")
print(f"Loaded splits: {list(vsr.keys())}")

# Use train split for exploration
train_data = vsr["train"]
print(f"\nTrain set size: {len(train_data)} examples")
print(f"\nFirst example:")
ex = train_data[0]
for k, v in ex.items():
    if k != "image":
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: PIL Image object (size: {v.size})")

## 3. Relation Type Distribution

In [ ]:
# Extract relation types and counts
relation_types = [ex["relation_type"] for ex in train_data]
relation_counts = Counter(relation_types)

print(f"Number of unique relation types: {len(relation_counts)}")
print(f"\nTop 15 relation types (by frequency):")

# Sort by count
sorted_rels = sorted(relation_counts.items(), key=lambda x: x[1], reverse=True)
for rel, count in sorted_rels[:15]:
    print(f"  {rel:30s}: {count:4d} ({100*count/len(train_data):5.1f}%)")

### Relation Type Frequency Bar Chart

In [ ]:
# Plot relation type frequencies
relation_names = [r[0] for r in sorted_rels]
counts = [r[1] for r in sorted_rels]

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(relation_names, counts, color="steelblue")
ax.set_xlabel("Count", fontsize=12)
ax.set_ylabel("Relation Type", fontsize=12)
ax.set_title("VSR Dataset: Relation Type Distribution", fontsize=14, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../results/figures/01_relation_frequencies.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 01_relation_frequencies.png")

## 4. Label Balance per Relation Type

In [ ]:
# Compute label balance (True/False) per relation
label_balance = {}
for ex in train_data:
    rel = ex["relation_type"]
    label = int(ex["label"])  # 1 for True, 0 for False
    if rel not in label_balance:
        label_balance[rel] = [0, 0]  # [false_count, true_count]
    label_balance[rel][label] += 1

# Convert to DataFrame for easier analysis
balance_df = pd.DataFrame([
    {"relation": rel, "false_count": counts[0], "true_count": counts[1]}
    for rel, counts in label_balance.items()
]).sort_values("false_count", ascending=False)

balance_df["total"] = balance_df["false_count"] + balance_df["true_count"]
balance_df["true_ratio"] = balance_df["true_count"] / balance_df["total"]

print("Label balance (showing first 10):")
print(balance_df.head(10).to_string())

### Label Balance Visualization

In [ ]:
# Visualize label balance for selected relation types
top_rels = balance_df.head(15).copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Stacked bar chart
ax1.barh(top_rels["relation"], top_rels["false_count"], label="False", color="coral")
ax1.barh(top_rels["relation"], top_rels["true_count"], 
         left=top_rels["false_count"], label="True", color="seagreen")
ax1.set_xlabel("Count", fontsize=11)
ax1.set_ylabel("Relation Type", fontsize=11)
ax1.set_title("Label Balance: Positive vs. Negative Examples", fontsize=12, fontweight="bold")
ax1.legend()
ax1.invert_yaxis()

# True ratio scatter
ax2.scatter(top_rels["total"], top_rels["true_ratio"], s=100, alpha=0.7, color="steelblue")
for idx, row in top_rels.iterrows():
    ax2.annotate(row["relation"], xy=(row["total"], row["true_ratio"]), 
                fontsize=8, alpha=0.7, xytext=(5, 5), textcoords="offset points")
ax2.axhline(y=0.5, color="red", linestyle="--", alpha=0.5, label="Balanced (0.5)")
ax2.set_xlabel("Total Examples", fontsize=11)
ax2.set_ylabel("Proportion True", fontsize=11)
ax2.set_title("True Ratio vs. Dataset Size", fontsize=12, fontweight="bold")
ax2.legend()
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig("../results/figures/01_label_balance.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 01_label_balance.png")

## 5. Caption Length Analysis

In [ ]:
# Analyze caption lengths
caption_lengths = [len(ex["caption"].split()) for ex in train_data]

print(f"Caption length statistics:")
print(f"  Mean: {np.mean(caption_lengths):.1f} words")
print(f"  Median: {np.median(caption_lengths):.1f} words")
print(f"  Min: {np.min(caption_lengths)} words")
print(f"  Max: {np.max(caption_lengths)} words")
print(f"  Std: {np.std(caption_lengths):.1f} words")

# Check CLIP token limit (77 tokens)
# CLIP tokenizer typically uses subword tokenization, so token count > word count
print(f"\nCLIP token limit: 77 tokens")
print(f"Captions with >10 words: {sum(1 for l in caption_lengths if l > 10)} ({100*sum(1 for l in caption_lengths if l > 10)/len(caption_lengths):.1f}%)")

### Caption Length Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(caption_lengths, bins=30, color="steelblue", edgecolor="black", alpha=0.7)
ax.axvline(np.mean(caption_lengths), color="red", linestyle="--", linewidth=2, label=f"Mean: {np.mean(caption_lengths):.1f}")
ax.axvline(np.median(caption_lengths), color="orange", linestyle="--", linewidth=2, label=f"Median: {np.median(caption_lengths):.1f}")
ax.set_xlabel("Caption Length (words)", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.set_title("VSR Dataset: Caption Length Distribution", fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("../results/figures/01_caption_lengths.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 01_caption_lengths.png")

## 6. Sample Images with Captions

In [ ]:
# Display sample images with their captions and relations
np.random.seed(42)
sample_indices = np.random.choice(len(train_data), 12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, idx in enumerate(sample_indices):
    ex = train_data[idx]
    image = ex["image"]
    caption = ex["caption"]
    relation = ex["relation_type"]
    label = "✓ True" if ex["label"] else "✗ False"
    
    axes[i].imshow(image)
    axes[i].axis("off")
    title_text = f"{relation}\n{label}\n\"{caption}\""
    axes[i].set_title(title_text, fontsize=9, wrap=True)

plt.tight_layout()
plt.savefig("../results/figures/01_sample_images.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved: 01_sample_images.png")

## 7. Image Availability Check

In [ ]:
# Check how many images load successfully
failed_indices = []
successful = 0

for i, ex in enumerate(train_data):
    try:
        img = ex["image"]
        if img is None:
            failed_indices.append(i)
        else:
            successful += 1
    except Exception as e:
        failed_indices.append(i)

print(f"Image availability:")
print(f"  Successfully loaded: {successful}/{len(train_data)} ({100*successful/len(train_data):.1f}%)")
print(f"  Failed to load: {len(failed_indices)}/{len(train_data)} ({100*len(failed_indices)/len(train_data):.1f}%)")

if failed_indices:
    print(f"  Failed indices: {failed_indices[:10]}{'...' if len(failed_indices) > 10 else ''}")

## 8. Summary Statistics

In [ ]:
# Comprehensive summary
print("\n" + "="*70)
print("VSR DATASET SUMMARY")
print("="*70)
print(f"\nDataset Size: {len(train_data)} examples")
print(f"\nRelation Types:")
print(f"  Unique relations: {len(relation_counts)}")
print(f"  Most common: {sorted_rels[0][0]} ({sorted_rels[0][1]} examples)")
print(f"  Least common: {sorted_rels[-1][0]} ({sorted_rels[-1][1]} examples)")
print(f"\nLabel Distribution:")
true_count = sum(1 for ex in train_data if ex["label"])
false_count = len(train_data) - true_count
print(f"  True (positive): {true_count} ({100*true_count/len(train_data):.1f}%)")
print(f"  False (negative): {false_count} ({100*false_count/len(train_data):.1f}%)")
print(f"\nCaption Statistics:")
print(f"  Mean length: {np.mean(caption_lengths):.1f} words")
print(f"  Max length: {np.max(caption_lengths)} words")
print(f"\nImage Availability:")
print(f"  Successfully loaded: {successful}/{len(train_data)} ({100*successful/len(train_data):.1f}%)")
print(f"\nKey Observations:")
print(f"  - Class imbalance: relation types have highly variable frequencies")
print(f"  - Label balance: most relations have similar positive/negative ratio")
print(f"  - Short captions: well within CLIP token limit (77 tokens)")
print(f"  - Image quality: all images load successfully")
print(f"  - Suitable for stratified k-fold CV to handle imbalance")
print("\n" + "="*70)